# Bronze to Silver: Geospatial Risk Intelligence Pipeline


> **Recommended runtime: Large** for the San Diego County AOI (~1.03M buildings, about 50 minutes; Medium ran the ~358K-building city in 13 min but lost executors in the KNN stage under a heavier load) — raster zonal stats + spatial KNN + cached-buildings shuffle.
This notebook transforms Bronze-layer Iceberg tables into Silver-layer tables containing
spatially joined, analytically useful datasets using Wherobots Cloud and Apache Sedona.

## Silver tables produced

| Silver Table | Operation | Inputs |
|---|---|---|
| `org_catalog.silver.asset_wildfire_exposure` | Zonal Statistics (raster → vector) | USFS Wildfire Burn Prob / Flame Length + Overture Buildings |
| `org_catalog.silver.asset_flood_exposure` | Zonal Statistics + Temporal Aggregation | OPERA DSWx-S1 (SAR flood) + Overture Buildings |
| `org_catalog.silver.asset_weather_density` | KNN Join + Buffer Aggregation | NOAA SWDI (hail, structure, tvs) + Overture Buildings |
| `org_catalog.silver.asset_enriched` | Conflation Join | All three above |

## Bronze inputs

| Dataset | Catalog Table | Type |
|---|---|---|
| Buildings | `wherobots_open_data.overture_maps_foundation.buildings_building` | Vector |
| Wildfire Burn Probability | `org_catalog.wildfire_risk.burn_probability_conus` | Raster |
| Wildfire Flame Length | `org_catalog.wildfire_risk.conditional_flame_length_conus` | Raster |
| OPERA DSWx-S1 Flood | `org_catalog.opera.dswx_s1` | Raster |
| NOAA SWDI Hail | `org_catalog.noaa_swdi.hail` | Vector |
| NOAA SWDI Structure | `org_catalog.noaa_swdi.structure` | Vector |
| NOAA SWDI TVS | `org_catalog.noaa_swdi.tvs` | Vector |

## Notebook sequence

```
raw-to-bronze.ipynb  →  bronze-to-silver.ipynb  →  silver-to-gold.ipynb
                             (you are here)
```

## Prerequisites
- Wherobots Cloud runtime with Apache Sedona
- Bronze tables populated (run `raw-to-bronze.ipynb` first)
- `wkls` Python library (`pip install wkls`)

## 0. Configuration & Session Setup

In [1]:
from sedona.spark import *
from pyspark.sql import functions as F
from datetime import datetime, date
import wkls

# ── Pipeline Parameters ──────────────────────────────────────────────────────
# Geographic scope: San Diego County, California via wkls (~1.03M buildings).
# This is the workshop's reference AOI: the Aurora seed and the lab pages describe
# a county run, and the wildland-urban interface (Poway, Ramona, Julian) is outside
# the city limits. For a faster ~358K-building run use wkls.us.ca.sandiego.wkt().
AOI_WKT = wkls.us.ca.sandiegocounty.wkt()
print(f"AOI loaded: San Diego County, California ({len(AOI_WKT):,} chars)")

# Temporal windows — per source, since datasets have different coverage periods
# OPERA DSWx-S1 flood (Sentinel-1 SAR, Dec 2025 – Mar 2026 storm season)
FLOOD_WINDOW_START = "2025-12-01"
FLOOD_WINDOW_END   = "2026-03-31"

# NOAA SWDI severe weather (hail, structure, TVS)
WEATHER_WINDOW_START = "2025-01-01"
WEATHER_WINDOW_END   = "2026-03-25"

# Wildfire risk classification thresholds (burn probability)
WF_THRESHOLD_EXTREME   = 0.05
WF_THRESHOLD_VERY_HIGH = 0.01
WF_THRESHOLD_HIGH      = 0.002
WF_THRESHOLD_MODERATE  = 0.0005

# ── Table References ─────────────────────────────────────────────────────────
# Bronze / Source tables
BUILDINGS_TABLE   = "wherobots_open_data.overture_maps_foundation.buildings_building"
WILDFIRE_BP_TABLE = "org_catalog.wildfire_risk.burn_probability_conus"
WILDFIRE_FL_TABLE = "org_catalog.wildfire_risk.conditional_flame_length_conus"
FLOOD_TABLE       = "org_catalog.opera.dswx_s1"
SWDI_HAIL_TABLE   = "org_catalog.noaa_swdi.hail"
SWDI_STRUCT_TABLE = "org_catalog.noaa_swdi.structure"
SWDI_TVS_TABLE    = "org_catalog.noaa_swdi.tvs"

# Silver output database (in org_catalog)
SILVER_DB = "silver"

print(f"Flood window:       {FLOOD_WINDOW_START} → {FLOOD_WINDOW_END}")
print(f"Weather window:     {WEATHER_WINDOW_START} → {WEATHER_WINDOW_END}")

AOI loaded: San Diego County, California (14,960 chars)
Flood window:       2025-12-01 → 2026-03-31
Weather window:     2025-01-01 → 2026-03-25


In [2]:
# ── Initialize Wherobots Sedona Session ───────────────────────────────────────
config = SedonaContext.builder().getOrCreate()
config.sparkContext.setLogLevel("ERROR")  # hide Spark WARN chatter (function registration, plan truncation, window partitioning); real errors still surface
sedona = SedonaContext.create(config)

# Create the Silver database if it doesn't exist
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{SILVER_DB}")

print("Sedona session initialized ✓")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/09/16 21:01:05 INFO core/src/lib.rs: Sedona native acceleration engine v0.15.3 ready


Sedona session initialized ✓


## 1. Explore Bronze Tables

Before transforming, verify the Bronze layer is populated and inspect row counts and schema.

In [3]:
# ── Inspect source tables ─────────────────────────────────────────────────────
source_tables = [
    BUILDINGS_TABLE,
    WILDFIRE_BP_TABLE,
    WILDFIRE_FL_TABLE,
    FLOOD_TABLE,
    SWDI_HAIL_TABLE,
    SWDI_STRUCT_TABLE,
    SWDI_TVS_TABLE,
]

for table in source_tables:
    try:
        df = sedona.table(table)
        count = df.count()
        print(f"  ✓ {table:70s} rows: {count:>12,}")
    except Exception as e:
        print(f"  ✗ {table:70s} ERROR: {e}")

  ✓ wherobots_open_data.overture_maps_foundation.buildings_building        rows: 2,529,582,613
  ✓ org_catalog.wildfire_risk.burn_probability_conus                       rows:      970,268
  ✓ org_catalog.wildfire_risk.conditional_flame_length_conus               rows:      970,268
  ✓ org_catalog.opera.dswx_s1                                              rows:    2,395,168
  ✓ org_catalog.noaa_swdi.hail                                             rows:   25,422,486
  ✓ org_catalog.noaa_swdi.structure                                        rows:   80,985,387
  ✓ org_catalog.noaa_swdi.tvs                                              rows:       91,387


In [4]:
# ── Load source tables scoped to AOI ──────────────────────────────────────────
# Buildings: filter by AOI polygon
buildings = sedona.sql(f"""
    SELECT id, geometry, height, num_floors, class, subtype, names.primary AS building_name
    FROM {BUILDINGS_TABLE}
    WHERE ST_Intersects(geometry, ST_GeomFromText('{AOI_WKT}'))
""")
buildings.createOrReplaceTempView("buildings")

# Wildfire rasters: filter tiles overlapping AOI
wildfire_bp = sedona.table(WILDFIRE_BP_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
wildfire_bp.createOrReplaceTempView("wildfire_bp")

wildfire_fl = sedona.table(WILDFIRE_FL_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
wildfire_fl.createOrReplaceTempView("wildfire_fl")

# OPERA DSWx-S1 Flood: load B01_WTR (water classification) band only, filtered to AOI
# B01_WTR values: 0=not water, 1=open water, 2=partial surface water
flood_wtr = sedona.table(FLOOD_TABLE).filter(
    f"band = 'B01_WTR' AND RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))"
)
flood_wtr.createOrReplaceTempView("flood_b01_wtr")

# SWDI tables are loaded per-event-type in section 4 (KNN join)

print(f"Buildings:      {buildings.count():>12,} rows (AOI)")
print(f"Wildfire BP:    {wildfire_bp.count():>12,} tiles")
print(f"Wildfire FL:    {wildfire_fl.count():>12,} tiles")
print(f"Flood B01_WTR:  {flood_wtr.count():>12,} tiles")

Buildings:         1,027,269 rows (AOI)
Wildfire BP:             824 tiles
Wildfire FL:             824 tiles
Flood B01_WTR:        67,539 tiles


## 2. Silver Table: `asset_wildfire_exposure`

**Operation**: Zonal Statistics (raster → vector)

For each building footprint, compute the mean and max burn probability from USFS wildfire risk rasters,
plus the mean conditional flame length. Classify each asset into a wildfire risk tier.

> **Sedona Functions**: `RS_ZonalStats(raster, geometry, statType)`, `RS_Intersects`
> 
> **Note**: RS_ZonalStats automatically handles CRS transformation between raster and geometry

In [5]:
# ── 2a. Compute wildfire exposure via zonal stats ────────────────────────────
# Join building footprints with wildfire raster tiles using RS_Intersects,
# then extract burn probability and flame length statistics per asset.
# RS_ZonalStats(raster, geometry, statType) handles CRS alignment automatically.

wildfire_exposure = sedona.sql(f"""
    WITH bp AS (
        SELECT
            b.id AS asset_id,
            b.geometry,
            b.height,
            b.num_floors,
            b.class,
            avg(RS_ZonalStats(w.raster, b.geometry, 1, 'mean', true)) AS burn_prob_mean,
            max(RS_ZonalStats(w.raster, b.geometry, 1, 'max', true))  AS burn_prob_max
        FROM buildings b
        JOIN wildfire_bp w
            ON RS_Intersects(w.raster, b.geometry)
        GROUP BY b.id, b.geometry, b.height, b.num_floors, b.class
    ),
    fl AS (
        SELECT
            b.id AS asset_id,
            avg(RS_ZonalStats(w.raster, b.geometry, 1, 'mean', true)) AS flame_length_mean
        FROM buildings b
        JOIN wildfire_fl w
            ON RS_Intersects(w.raster, b.geometry)
        GROUP BY b.id
    )
    SELECT
        bp.asset_id,
        'building' AS asset_type,
        bp.geometry,
        bp.height,
        bp.num_floors,
        bp.class,
        bp.burn_prob_mean,
        bp.burn_prob_max,
        COALESCE(fl.flame_length_mean, 0.0) AS flame_length_mean,
        CASE
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_EXTREME}   THEN 'extreme'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_VERY_HIGH} THEN 'very_high'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_HIGH}      THEN 'high'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_MODERATE}  THEN 'moderate'
            ELSE 'low'
        END AS wildfire_risk_class,
        current_timestamp() AS computed_at
    FROM bp
    LEFT JOIN fl ON bp.asset_id = fl.asset_id
""")

WILDFIRE_SILVER = f"org_catalog.{SILVER_DB}.asset_wildfire_exposure"
wildfire_exposure.writeTo(WILDFIRE_SILVER).createOrReplace()
wildfire_exposure = sedona.table(WILDFIRE_SILVER)

print(f"Wildfire exposure rows: {wildfire_exposure.count():,}")
wildfire_exposure.groupBy("wildfire_risk_class").count().orderBy("count", ascending=False).show()

Wildfire exposure rows: 1,027,269


,wildfire_risk_class,count
0,low,798586
1,moderate,82732
2,high,75911
3,very_high,60401
4,extreme,9639


## 3. Silver Table: `asset_flood_exposure`

**Operation**: Zonal Statistics, one week at a time → append to Iceberg (raster → vector)

For each building, compute the max DSWx-S1 **B01_WTR** classification per ISO week using
`allTouched=true` so buildings smaller than 30 m still capture intersecting flood pixels.

The loop processes **one week per iteration**, writing results to Iceberg via
`createOrReplace` (first week) then `append` (subsequent weeks). This keeps shuffle small
(only one week of raster tiles at a time) and gives free checkpointing — if a week fails,
restart from there.

The output is one row per **(asset, week)** — temporal rollup is deferred to the Gold layer.

| Column | Description |
|---|---|
| `flood_week` | Start of the ISO week |
| `flood_max_wtr_class` | Max WTR class that week (0=dry, 1=open water, 2=partial) |

> **Sedona Functions**: `RS_ZonalStats(raster, geometry, band, statType, excludeNoData)`,
> `RS_Intersects`

In [6]:
sedona.sql('SELECT RS_Metadata(raster) as metadata FROM flood_b01_wtr').show(5, 0)

,metadata
0,"Row(upperLeftX=495960.0, upperLeftY=3623220.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=512, tileHeight=512)"
1,"Row(upperLeftX=484440.0, upperLeftY=3684660.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=512, tileHeight=512)"
2,"Row(upperLeftX=499800.0, upperLeftY=3623220.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=512, tileHeight=512)"
3,"Row(upperLeftX=523020.0, upperLeftY=3696180.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=512, tileHeight=512)"
4,"Row(upperLeftX=538380.0, upperLeftY=3646260.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=512, tileHeight=512)"


In [7]:
# ── 3a. Compute weekly flood exposure, one week at a time → Iceberg ───────────
# Process each ISO week independently to keep shuffle small (buildings × 1 week
# of raster tiles). Results are appended to Iceberg week by week — free
# checkpointing if a week fails.
# allTouched=true (5th arg) ensures partially-overlapping pixels are included.
# B01_WTR water classification: 0=dry, 1=open water, 2=partial surface water

from datetime import date, timedelta

FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"

# Generate ISO week start dates covering the flood window
start = date.fromisoformat(FLOOD_WINDOW_START)
end   = date.fromisoformat(FLOOD_WINDOW_END)

# Align to Monday (ISO week start)
week_start = start - timedelta(days=start.weekday())
weeks = []
while week_start <= end:
    weeks.append(week_start)
    week_start += timedelta(days=7)

print(f"Flood window: {FLOOD_WINDOW_START} → {FLOOD_WINDOW_END}")
print(f"Processing {len(weeks)} weeks: {weeks[0]} → {weeks[-1]}\n")

first = True
for i, wk in enumerate(weeks):
    wk_end = wk + timedelta(days=6)
    # Clamp to the actual flood window boundaries
    wk_start_clamped = max(wk, start)
    wk_end_clamped   = min(wk_end, end)

    df = sedona.sql(f"""
        SELECT
            b.id                                                        AS asset_id,
            'building'                                                  AS asset_type,
            b.geometry,
            DATE('{wk.isoformat()}')                                    AS flood_week,
            MAX(RS_ZonalStats(w.raster, b.geometry, 1, 'max', true))    AS flood_max_wtr_class,
            DATE('{FLOOD_WINDOW_START}')                                AS observation_window_start,
            DATE('{FLOOD_WINDOW_END}')                                  AS observation_window_end,
            current_timestamp()                                         AS computed_at
        FROM buildings b
        JOIN flood_b01_wtr w
            ON RS_Intersects(w.raster, b.geometry)
        WHERE w.acq_date BETWEEN DATE('{wk_start_clamped.isoformat()}')
                              AND DATE('{wk_end_clamped.isoformat()}')
        GROUP BY b.id, b.geometry
    """)

    if first:
        df.writeTo(FLOOD_SILVER).createOrReplace()
        first = False
    else:
        df.writeTo(FLOOD_SILVER).append()

    print(f"  ✓ week {i+1}/{len(weeks)}  {wk_start_clamped} → {wk_end_clamped}")

total = sedona.table(FLOOD_SILVER).count()
print(f"\n✓ Wrote {total:,} rows to {FLOOD_SILVER}  (asset × week)")

Flood window: 2025-12-01 → 2026-03-31
Processing 18 weeks: 2025-12-01 → 2026-03-30

  ✓ week 1/18  2025-12-01 → 2025-12-07
  ✓ week 2/18  2025-12-08 → 2025-12-14
  ✓ week 3/18  2025-12-15 → 2025-12-21
  ✓ week 4/18  2025-12-22 → 2025-12-28
  ✓ week 5/18  2025-12-29 → 2026-01-04
  ✓ week 6/18  2026-01-05 → 2026-01-11
  ✓ week 7/18  2026-01-12 → 2026-01-18
  ✓ week 8/18  2026-01-19 → 2026-01-25
  ✓ week 9/18  2026-01-26 → 2026-02-01
  ✓ week 13/18  2026-02-23 → 2026-03-01
  ✓ week 14/18  2026-03-02 → 2026-03-08
  ✓ week 15/18  2026-03-09 → 2026-03-15
  ✓ week 16/18  2026-03-16 → 2026-03-22
  ✓ week 17/18  2026-03-23 → 2026-03-29
  ✓ week 18/18  2026-03-30 → 2026-03-31

✓ Wrote 17,463,573 rows to org_catalog.silver.asset_flood_exposure  (asset × week)


In [8]:
# ── 3b. Verify silver.asset_flood_exposure ────────────────────────────────────
FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"

flood_check = sedona.table(FLOOD_SILVER)
print(f"Rows:           {flood_check.count():,}")
print(f"Distinct assets:{flood_check.select('asset_id').distinct().count():,}")
print(f"Distinct weeks: {flood_check.select('flood_week').distinct().count():,}")
flood_check.groupBy("flood_week").count().orderBy("flood_week").show(50, truncate=False)

Rows:           17,463,573
Distinct assets:1,027,269
Distinct weeks: 17


,flood_week,count
0,2025-12-01,1027269
1,2025-12-08,1027269
2,2025-12-15,1027269
3,2025-12-22,1027269
4,2025-12-29,1027269
5,2026-01-05,1027269
6,2026-01-12,1027269
7,2026-01-19,1027269
8,2026-01-26,1027269
9,2026-02-02,1027269


## 4. Silver Table: `asset_weather_density`

**Operation**: KNN Join with search radius + Spheroidal Distance (vector → vector)

For each building, find the K nearest NOAA SWDI severe weather events **per event type**
(hail, structure, TVS) using WherobotsDB's KNN join with `search_radius` to cap the search
at 25 km. Uses `use_sphere = true` so both the KNN search and the radius are in meters.

> **WherobotsDB**: `ST_KNN(R, S, k, use_sphere, search_radius)` — 5th param limits search distance
>
> **Sources**: `org_catalog.noaa_swdi.hail`, `.structure`, `.tvs` — queried separately per event type

In [9]:
# ── 4a. KNN join per SWDI event type → append to Iceberg ─────────────────────
# Use ST_KNN with search_radius (Wherobots extension) to limit neighbor search
# to 25 km. use_sphere=TRUE means both the KNN distance and search_radius are
# in meters. Pre-filter each SWDI table to AOI + temporal window first, and
# MATERIALIZE the filtered subset to an Iceberg staging table before the join:
# Sedona warns that a filter pushed down on the object side of a KNN join
# ("JoinQueryDetector: Filter pushdown detected on the object side of a KNN
# join") can return incorrect neighbours. Joining against a materialized table
# leaves nothing to push down.

K_NEAREST      = 10
SEARCH_RADIUS  = 25000    # 25 km in meters (search_radius param)
BUFFER_NEAR_M  = 5000     # 5 km threshold for aggregation
BUFFER_FAR_M   = 25000    # 25 km threshold for aggregation

KNN_STAGING = f"org_catalog.{SILVER_DB}.weather_knn_staging"

swdi_sources = {
    "hail":      (SWDI_HAIL_TABLE,   "SEVPROB",     "MAXSIZE"),
    "structure": (SWDI_STRUCT_TABLE,  "MAX_REFLECT", "VIL"),
    "tvs":       (SWDI_TVS_TABLE,    "MXDV",        "MAX_SHEAR"),
}

first = True
for event_type, (table, severity_col, magnitude_col) in swdi_sources.items():
    # Step 1: pre-filter to AOI + temporal window and materialize to Iceberg
    # (replaced on every run), then join against the materialized table.
    aoi_table = f"org_catalog.{SILVER_DB}.swdi_{event_type}_aoi"
    sedona.sql(f"""
        SELECT geometry, ZTIME, {severity_col}, {magnitude_col}
        FROM {table}
        WHERE ZTIME BETWEEN TIMESTAMP('{WEATHER_WINDOW_START}') AND TIMESTAMP('{WEATHER_WINDOW_END}')
          AND ST_Intersects(geometry, ST_GeomFromText('{AOI_WKT}'))
    """).writeTo(aoi_table).createOrReplace()
    view_name = f"swdi_{event_type}_aoi"
    sedona.table(aoi_table).createOrReplaceTempView(view_name)
    cnt = sedona.sql(f"SELECT COUNT(*) AS c FROM {view_name}").collect()[0]["c"]
    print(f"  {event_type}: {cnt:,} events in AOI (materialized to {aoi_table})")

    # Step 2: KNN join with use_sphere=TRUE and search_radius=25km
    df = sedona.sql(f"""
        SELECT
            b.id                                              AS asset_id,
            b.geometry                                        AS asset_geometry,
            '{event_type}'                                    AS event_type,
            e.ZTIME                                           AS event_time,
            e.{severity_col}                                  AS severity,
            e.{magnitude_col}                                 AS magnitude,
            ST_DistanceSpheroid(b.geometry, e.geometry)        AS distance_m
        FROM buildings b
        JOIN {view_name} e
            ON ST_KNN(b.geometry, e.geometry, {K_NEAREST}, true, {SEARCH_RADIUS})
    """)

    # Step 3: write to Iceberg — createOrReplace first, append after
    if first:
        df.writeTo(KNN_STAGING).createOrReplace()
        first = False
    else:
        df.writeTo(KNN_STAGING).append()
    print(f"  ✓ {event_type} written to {KNN_STAGING}")

total = sedona.table(KNN_STAGING).count()
print(f"\nTotal staging rows: {total:,}")
sedona.table(KNN_STAGING).show(10, truncate=False)

  hail: 1,321 events in AOI (materialized to org_catalog.silver.swdi_hail_aoi)
  ✓ hail written to org_catalog.silver.weather_knn_staging
  structure: 9,519 events in AOI (materialized to org_catalog.silver.swdi_structure_aoi)
  ✓ structure written to org_catalog.silver.weather_knn_staging
  tvs: 2 events in AOI (materialized to org_catalog.silver.swdi_tvs_aoi)
  ✓ tvs written to org_catalog.silver.weather_knn_staging

Total staging rows: 20,848,693


,asset_id,asset_geometry,event_type,event_time,severity,magnitude,distance_m
0,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-11-18 04:13:31,40.0,0.0,393.0879137391326
1,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-12-27 01:55:47,48.0,2.0,393.0879137391326
2,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-01-27 04:14:18,47.0,1.0,393.0879137391326
3,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-11-15 15:53:37,44.0,3.0,732.0668832551793
4,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-11-21 01:16:57,43.0,1.0,732.0668832551793
5,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-12-25 00:01:41,48.0,3.0,2147.181259739652
6,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-09-17 22:32:43,48.0,7.0,2270.2012158711827
7,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-11-18 04:08:19,40.0,0.0,2309.233863598839
8,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-01-27 04:08:13,46.0,2.0,2309.233863598839
9,ae1c65ee-de08-41bb-b7ce-2cf068d14d73,,structure,2025-03-06 05:36:41,42.0,1.0,2502.763089657969


In [10]:
# ── 4b. Aggregate KNN staging → asset_weather_density ────────────────────────
# Read from the Iceberg staging table (no in-memory cache needed).

WEATHER_SILVER = f"org_catalog.{SILVER_DB}.asset_weather_density"

weather_density = sedona.sql(f"""
    SELECT
        asset_id,
        'building'                                                 AS asset_type,
        asset_geometry                                             AS geometry,

        SUM(CASE WHEN distance_m <= {BUFFER_NEAR_M} THEN 1 ELSE 0 END)   AS event_count_5km,
        SUM(CASE WHEN distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END)    AS event_count_25km,

        MIN(distance_m)                                            AS nearest_event_dist_m,

        MIN(CASE WHEN event_type = 'hail'      THEN distance_m END) AS nearest_hail_m,
        MIN(CASE WHEN event_type = 'structure'  THEN distance_m END) AS nearest_structure_m,
        MIN(CASE WHEN event_type = 'tvs'        THEN distance_m END) AS nearest_tvs_m,

        SUM(CASE WHEN event_type = 'hail'      AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS hail_count_25km,
        SUM(CASE WHEN event_type = 'structure'  AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS structure_count_25km,
        SUM(CASE WHEN event_type = 'tvs'        AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS tvs_count_25km,

        -- Per-event-type max severity (units differ: SEVPROB 0-100, MAX_REFLECT dBZ, MXDV knots)
        MAX(CASE WHEN event_type = 'hail'      THEN severity END)  AS max_hail_sevprob,
        MAX(CASE WHEN event_type = 'structure'  THEN severity END) AS max_structure_reflectivity,
        MAX(CASE WHEN event_type = 'tvs'        THEN severity END) AS max_tvs_delta_v,

        DATE('{WEATHER_WINDOW_START}')                             AS observation_window_start,
        DATE('{WEATHER_WINDOW_END}')                               AS observation_window_end,
        current_timestamp()                                        AS computed_at

    FROM {KNN_STAGING}
    GROUP BY asset_id, asset_geometry
""")

weather_density.writeTo(WEATHER_SILVER).createOrReplace()
row_count = sedona.table(WEATHER_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {WEATHER_SILVER}")

✓ Wrote 1,027,269 rows to org_catalog.silver.asset_weather_density


## 5. Silver Table: `asset_enriched`

**Operation**: Conflation Join — Merge all three hazard exposure layers onto the base asset table

This is the unified Silver table that Gold will consume. Each row is an asset with all hazard signals attached. Uses `LEFT JOIN` so assets with no exposure in a given hazard still appear (with nulls).

In [11]:
# ── 5a. Conflate wildfire + weather exposure onto base assets ─────────────────
# LEFT JOIN ensures every building in scope appears even if it has no
# wildfire/weather exposure — those columns will be null.
#
# NOTE: Flood exposure (asset_flood_exposure) is NOT joined here because it
# contains weekly rows (one per asset × week). The Gold layer joins and
# aggregates flood data directly to preserve temporal granularity for scoring.

WILDFIRE_SILVER = f"org_catalog.{SILVER_DB}.asset_wildfire_exposure"
WEATHER_SILVER  = f"org_catalog.{SILVER_DB}.asset_weather_density"

asset_enriched = sedona.sql(f"""
    SELECT
        b.id                               AS asset_id,
        'building'                         AS asset_type,
        b.geometry,
        b.class                            AS building_class,
        b.height,
        b.num_floors,

        -- Wildfire exposure
        w.burn_prob_mean,
        w.burn_prob_max,
        w.flame_length_mean,
        w.wildfire_risk_class,

        -- Severe weather density (KNN-derived)
        s.event_count_5km,
        s.event_count_25km,
        s.nearest_event_dist_m,
        s.nearest_hail_m,
        s.nearest_structure_m,
        s.nearest_tvs_m,
        s.hail_count_25km,
        s.structure_count_25km,
        s.tvs_count_25km,
        s.max_hail_sevprob,
        s.max_structure_reflectivity,
        s.max_tvs_delta_v,

        -- Data coverage flags (null = no data, not zero risk)
        CASE WHEN w.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_wildfire_data,
        CASE WHEN s.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_weather_data,

        -- Temporal window
        DATE('{WEATHER_WINDOW_START}')     AS weather_window_start,
        DATE('{WEATHER_WINDOW_END}')       AS weather_window_end,

        current_timestamp()                AS computed_at

    FROM buildings b
    LEFT JOIN {WILDFIRE_SILVER} w
        ON b.id = w.asset_id
    LEFT JOIN {WEATHER_SILVER} s
        ON b.id = s.asset_id
""")

asset_enriched.cache()
print(f"Asset enriched rows: {asset_enriched.count():,}")
asset_enriched.printSchema()

Asset enriched rows: 1,027,269
root
 |-- asset_id: string (nullable = true)
 |-- asset_type: string (nullable = false)
 |-- geometry: geometry (nullable = true)
 |-- building_class: string (nullable = true)
 |-- height: double (nullable = true)
 |-- num_floors: integer (nullable = true)
 |-- burn_prob_mean: double (nullable = true)
 |-- burn_prob_max: double (nullable = true)
 |-- flame_length_mean: double (nullable = true)
 |-- wildfire_risk_class: string (nullable = true)
 |-- event_count_5km: long (nullable = true)
 |-- event_count_25km: long (nullable = true)
 |-- nearest_event_dist_m: double (nullable = true)
 |-- nearest_hail_m: double (nullable = true)
 |-- nearest_structure_m: double (nullable = true)
 |-- nearest_tvs_m: double (nullable = true)
 |-- hail_count_25km: long (nullable = true)
 |-- structure_count_25km: long (nullable = true)
 |-- tvs_count_25km: long (nullable = true)
 |-- max_hail_sevprob: double (nullable = true)
 |-- max_structure_reflectivity: double (nullable

In [12]:
# ── 5b. Quick data quality check ──────────────────────────────────────────────
# Show coverage: what % of assets have non-null values for each hazard layer.
# Flood coverage is checked separately since it lives in its own weekly table.

total = asset_enriched.count()
wf_coverage = asset_enriched.filter(F.col("has_wildfire_data")).count()
sw_coverage = asset_enriched.filter(F.col("has_weather_data")).count()

FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"
fl_assets = sedona.sql(f"SELECT COUNT(DISTINCT asset_id) AS c FROM {FLOOD_SILVER}").collect()[0]["c"]

print(f"Total assets:                  {total:>10,}")
print(f"Wildfire exposure coverage:    {wf_coverage:>10,}  ({wf_coverage/total*100:.1f}%)")
print(f"Flood exposure coverage:       {fl_assets:>10,}  ({fl_assets/total*100:.1f}%)  (from weekly table)")
print(f"Severe weather coverage:       {sw_coverage:>10,}  ({sw_coverage/total*100:.1f}%)")
print()

# Distribution of wildfire risk classes
print("Wildfire risk class distribution:")
asset_enriched.groupBy("wildfire_risk_class").count().orderBy("count", ascending=False).show()

Total assets:                   1,027,269
Wildfire exposure coverage:     1,027,269  (100.0%)
Flood exposure coverage:        1,027,269  (100.0%)  (from weekly table)
Severe weather coverage:        1,027,269  (100.0%)

Wildfire risk class distribution:


,wildfire_risk_class,count
0,low,798586
1,moderate,82732
2,high,75911
3,very_high,60401
4,extreme,9639


In [13]:
# ── 5c. Write silver.asset_enriched to Iceberg ───────────────────────────────
ENRICHED_SILVER = f"org_catalog.{SILVER_DB}.asset_enriched"

asset_enriched.writeTo(ENRICHED_SILVER).createOrReplace()

row_count = sedona.table(ENRICHED_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {ENRICHED_SILVER}")

asset_enriched.unpersist()  # release cached DataFrame from memory


✓ Wrote 1,027,269 rows to org_catalog.silver.asset_enriched


,asset_id,asset_type,geometry,building_class,height,num_floors,burn_prob_mean,burn_prob_max,flame_length_mean,wildfire_risk_class,event_count_5km,event_count_25km,nearest_event_dist_m,nearest_hail_m,nearest_structure_m,nearest_tvs_m,hail_count_25km,structure_count_25km,tvs_count_25km,max_hail_sevprob,max_structure_reflectivity,max_tvs_delta_v,has_wildfire_data,has_weather_data,weather_window_start,weather_window_end,computed_at
0,3f9dbb1d-1f26-44cc-8ec0-f99e20dd3001,building,,,5.546636581420898,nan,0.01946364901959896,0.01946364901959896,2.503675937652588,very_high,12,21,808.9016543862873,3962.3487380289944,808.9016543862873,16086.45056345328,10,10,1,0.0,53.0,63.0,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
1,2693c5c1-b005-4029-ab71-6a11ece85813,building,,,nan,nan,0.002270636541652493,0.002807593671604991,2.6834364533424377,high,13,20,809.692003444093,1411.0440433941978,809.692003444093,nan,10,10,0,30.0,47.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
2,2fbfc609-9a9f-4bf1-97be-4e089659ab70,building,,,nan,nan,0.03306810185313225,0.033087827265262604,4.318093299865723,very_high,10,20,692.9293738965939,6340.117043837218,692.9293738965939,nan,10,10,0,50.0,46.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
3,82d1137b-751b-408d-9c47-fc2dfc1b66fe,building,,,nan,nan,0.03300122730433941,0.03310862183570862,5.481401443481445,very_high,10,20,673.127932149831,6405.266569986521,673.127932149831,nan,10,10,0,50.0,46.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
4,c58cc7c0-9595-41a3-be55-117ee8e66595,building,,,nan,nan,0.0,0.0,0.0,low,14,20,460.3938339411624,460.3938339411624,460.3938339411624,nan,10,10,0,0.0,52.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
5,01615c4a-3ccb-4b8d-9f4a-e13ed1f342c0,building,,,4.221736907958984,nan,0.0,0.0,0.0,low,10,20,255.3159309342955,8483.56765557942,255.3159309342955,nan,10,10,0,0.0,51.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
6,28396008-5a65-4e40-805d-60f9ce7d74bc,building,,,nan,nan,0.0,0.0,0.0,low,14,20,426.01496713609896,426.01496713609896,426.01496713609896,nan,10,10,0,0.0,50.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
7,35b34a6c-ad3e-478d-ad41-94c942feb27f,building,,,nan,nan,0.013483853545039892,0.016929171979427338,1.031680703163147,very_high,12,21,959.7727704675384,4098.3435913144785,959.7727704675384,16014.799784321845,10,10,1,0.0,53.0,63.0,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
8,7cdae126-fabb-4d32-92b7-54677a0bae13,building,,,4.966979026794434,nan,0.0,0.0,0.0,low,10,20,228.0621657544075,8469.312151516096,228.0621657544075,nan,10,10,0,0.0,51.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806
9,d6ae592d-592e-42d8-b7a7-50cf35d9919d,building,"<path fill-rule=""evenodd"" fill=""#66cc99"" stroke=""#555555"" stroke-width=""2.2263120000047822e-05"" opacity=""0.6"" d=""M -117.2230722,32.726766 L -117.2232414,32.7268678 L -117.2235916,32.7264451 L -117.223593,32.7265169 L -117.223707,32.7265158 L -117.2237107,32.7259601 L -117.2237633,32.7259607 L -117.2237633,32.7259232 L -117.2237633,32.7258512 L -117.2237633,32.7258371 L -117.2235961,32.7258377 L -117.2235961,32.7258436 L -117.2235489,32.7258436 L -117.2235481,32.7259237 L -117.2235945,32.7259243 L -117.2235911,32.7263678 L -117.2235344,32.7263362 L -117.2232458,32.7266576 L -117.2229469,32.726471 L -117.2228838,32.7265573 L -117.2231239,32.7267048 L -117.2230722,32.726766 z"" />",,4.954677104949951,nan,0.0,0.0,0.0,low,10,20,453.64800420560465,10236.12761785135,453.64800420560465,nan,10,10,0,30.0,48.0,nan,True,True,2025-01-01,2026-03-25,2026-09-16 21:51:50.728806


## 6. Verify Silver Layer

Final verification: list all Silver tables and confirm row counts.

In [14]:
# ── Verify all Silver tables ──────────────────────────────────────────────────
silver_tables = [
    "asset_wildfire_exposure",
    "asset_flood_exposure",
    "asset_weather_density",
    "asset_enriched",
]

print("Silver Layer Summary")
print("=" * 72)
for table in silver_tables:
    fqn = f"org_catalog.{SILVER_DB}.{table}"
    df = sedona.table(fqn)
    count = df.count()
    cols = len(df.columns)
    print(f"  {fqn:55s}  {count:>10,} rows  {cols:>3} cols")
print("=" * 72)
print("\n✓ Bronze → Silver pipeline complete.")
print(f"  Next step: run silver-to-gold.ipynb to produce industry-specific Gold tables.")

Silver Layer Summary
  org_catalog.silver.asset_wildfire_exposure                1,027,269 rows   11 cols
  org_catalog.silver.asset_flood_exposure                  17,463,573 rows    8 cols
  org_catalog.silver.asset_weather_density                  1,027,269 rows   18 cols
  org_catalog.silver.asset_enriched                         1,027,269 rows   27 cols

✓ Bronze → Silver pipeline complete.
  Next step: run silver-to-gold.ipynb to produce industry-specific Gold tables.
